# 🫀 HeartGuard: Exploratory Data Analysis (EDA)
### End-to-End Cardiovascular Risk Prediction Pipeline

**Objective**: Perform comprehensive Exploratory Data Analysis on the Cleveland Heart Disease Dataset to identify key clinical risk factors, evaluate feature distributions, detect anomalies/outliers, analyze feature correlations, and formulate data transformation & feature engineering strategies.

--- 
## 1. Setup & Environment Initialisation

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Setting plotting aesthetic
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

# Ensure path to data
data_path = "../data/raw/heart.csv"
if not os.path.exists(data_path):
    data_path = "data/raw/heart.csv"

df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

--- 
## 2. Dataset Overview & Data Quality Checks

In [ ]:
# Information on data types and null values
print("=== DATASET INFO ===")
df.info()

print("\n=== MISSING VALUES ===")
print(df.isnull().sum())

print("\n=== DUPLICATE ROWS ===")
print(f"Duplicates count: {df.duplicated().sum()}")

print("\n=== SUMMARY STATISTICS ===")
df.describe().T

--- 
## 3. Target Variable Analysis (`target`)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Frequency plot
sns.countplot(data=df, x='target', palette=['#10B981', '#EF4444'], ax=ax[0])
ax[0].set_title("Heart Disease Class Distribution")
ax[0].set_xticklabels(["0: Low Risk (Normal)", "1: High Risk (Disease)"])
ax[0].set_ylabel("Patient Count")

# Percentage pie chart
target_counts = df['target'].value_counts()
ax[1].pie(target_counts, labels=["High Risk (1)", "Low Risk (0)"], autopct='%1.1f%%', 
        colors=['#EF4444', '#10B981'], startangle=140, explode=(0.05, 0))
ax[1].set_title("Percentage Breakdown")

plt.tight_layout()
plt.show()

print(f"Class proportions:\n{df['target'].value_counts(normalize=True)*100}")

**Key Finding**: The dataset is well-balanced (~54% High Risk vs ~46% Low Risk), requiring no aggressive resampling like SMOTE.

--- 
## 4. Continuous Features Analysis & Outlier Detection

In [ ]:
num_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

fig, axes = plt.subplots(len(num_cols), 2, figsize=(15, 18))

for i, col in enumerate(num_cols):
    # Distribution plot
    sns.histplot(df, x=col, hue='target', kde=True, element='step', 
                 palette=['#10B981', '#EF4444'], ax=axes[i, 0])
    axes[i, 0].set_title(f'Distribution of {col} by Target')
    
    # Boxplot for outliers
    sns.boxplot(df, x='target', y=col, palette=['#10B981', '#EF4444'], ax=axes[i, 1])
    axes[i, 1].set_title(f'Outliers & Quartiles of {col}')
    axes[i, 1].set_xticklabels(["Low Risk", "High Risk"])

plt.tight_layout()
plt.show()

--- 
## 5. Categorical Feature Relationships with Heart Disease

In [ ]:
cat_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
fig, axes = plt.subplots(4, 2, figsize=(16, 18))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    sns.countplot(data=df, x=col, hue='target', palette=['#10B981', '#EF4444'], ax=axes[i])
    axes[i].set_title(f'{col.upper()} vs Heart Disease Risk')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Count")
    axes[i].legend(["Low Risk", "High Risk"], loc='upper right')

plt.tight_layout()
plt.show()

--- 
## 6. Correlation Matrix & Feature Relationships

In [ ]:
plt.figure(figsize=(14, 10))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", 
            vmin=-1, vmax=1, linewidths=0.5, cbar_kws={"shrink": .8})
plt.title("Correlation Matrix of Clinical Features", fontsize=16)
plt.show()

print("Top correlations with Target:")
print(corr['target'].sort_values(ascending=False))

--- 
## 7. Key Findings & Engineering Recommendations

1. **Chest Pain (`cp`) & Max Heart Rate (`thalach`)**: Strongly positively correlated with heart disease. Non-anginal and atypical chest pain patients present significantly higher risk than asymptomatic ones.
2. **Exercise-Induced Angina (`exang`), ST Depression (`oldpeak`), & Vessels (`ca`)**: Strongly negatively correlated. Higher ST depression and higher number of colored major vessels strongly indicate severe ischemic disease.
3. **Domain Interaction Features Introduced**:
   - **`age_thalach_ratio`**: Captures heart rate capacity relative to age (`thalach / age`).
   - **`bp_chol_prod`**: Cardiovascular load interaction (`trestbps * chol`).
   - **`st_slope_risk`**: Combined ischemic stress marker (`oldpeak * slope`).
   - **`risk_score_sum`**: High-risk categorical indicator accumulator (`(cp > 0) + exang + (ca > 0)`).